# FBMP · Moment, shear & torsion envelopes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/FBMP_Envelope_Extractor.ipynb)

**Standalone Google Colab tool — upload your FB-MultiPier XML and download the envelopes.**

1. Select **Runtime → Run all**. No GPU is needed.
2. Choose one or more analyzed FBMP `.xml` files when prompted.
3. Read the **Strength** and **Service** tables displayed in the notebook. Every available limit state has separate M2, M3, V2, V3, and torsion results. Download either table as CSV or get the full ZIP.

The notebook contains all of its extraction code. It needs no companion files, Drive mount, API key, or FBMP installation. Colab's standard Python packages are used. Files are uploaded to your Colab runtime. If you upload several files, each is processed separately.

**Outputs:** M2, M3, V2, V3, and torsion T, with signed minimum/maximum, maximum absolute magnitude, units, and governing load cases/combinations. Detailed results retain element IDs, I/J ends, nodes, time steps, ties, and concurrent forces (including axial P).

**Strength and Service are separate:** the tables are computed from the detailed case results, grouped by each exact limit state. Service I, II, III, etc. remain separate when exported; no Service values are inferred from Strength. In the supplied file, **Service I is load case 11, combination 4**.

**Two distinct result sources:** the visible Strength/Service tables and explorer compute envelopes from original XML element-end forces, preserving their signs. The **Original FBMP summary** tab copies `OUTPUT_SUMMARY`, an overall envelope in which Strength cases may govern. Native summaries cannot be re-filtered from the governing cases alone. The tool flags magnitude differences between the two sources and does not attempt to reproduce FBMP's design sign adjustments.

Tested with `Pier_MinTip.XML`, FB-MultiPier 6.1.0: 4 analyzed cases, 190 exported elements, and 1,520 element-end records. Its pier-cap M2 and V3 summary magnitudes differ from the detailed force block; both are retained in **Data checks**.


In [ ]:
#@title Load the standalone extractor
"""FB-MultiPier XML envelope extraction. Standard library only; embedded in the notebook."""
from collections import defaultdict, Counter
from datetime import datetime, timezone
from pathlib import Path
import csv
import hashlib
import html
import io
import json
import math
import re
import zipfile
import xml.etree.ElementTree as ET

VERSION = "1.1.0"
COMPONENTS = {"M2": "MOMENT-2", "M3": "MOMENT-3", "V2": "SHEAR-2", "V3": "SHEAR-3", "T": "TORQUE"}
FORCES = {**COMPONENTS, "P": "AXIAL"}
LABELS = {"M2": "Moment about local 2", "M3": "Moment about local 3", "V2": "Shear in local 2", "V3": "Shear in local 3", "T": "Torsion"}
ITEM_MAP = {"moment about 2 axis": "M2", "moment about 3 axis": "M3",
            "shear in 2 direction": "V2", "shear in 3 direction": "V3",
            "torque": "T", "torsional force": "T"}
SIGN_NOTE = ("Reported FBMP envelopes are copied from OUTPUT_SUMMARY. Detailed envelopes use the XML's "
             "original element-local I/J end-force signs. No signs are reversed, axes rotated, or load factors applied. "
             "These two outputs may differ in signs and member scope; they are not interchangeable.")
METHOD_NOTE = ("Min and max are algebraic extremes. Absolute maximum is a nonnegative magnitude, with the "
               "signed governing value retained. Components are enveloped independently; their maxima need not coexist. "
               "Concurrent forces on detailed governing rows come from the same element end, case, combination, and time step. "
               "Only exported element ends are considered; interior peaks are not inferred. Units are retained exactly and "
               "different unit labels are never mixed. No new combinations or analysis are performed.")


class FBMPError(ValueError):
    pass


def _number(text, where):
    try:
        value = float((text or "").strip().replace("D", "E").replace("d", "e"))
    except (ValueError, TypeError):
        raise FBMPError(f"Invalid or missing numeric value at {where}: {text!r}") from None
    if not math.isfinite(value):
        raise FBMPError(f"Non-finite numeric value at {where}: {text!r}")
    return value


def _text(e, tag):
    return (e.findtext(tag) or "").strip()


def _nat(value):
    return [(0, int(p)) if p.isdigit() else (1, p.lower()) for p in re.split(r"(\d+)", str(value))]


def _identity(e):
    return e.get("number", e.get("id", ""))


def read_fbmp(data, filename="model.xml"):
    """Return raw forces, native summary, cases, and explicit diagnostic messages."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    if len(data) > 200 * 1024 * 1024:
        raise FBMPError("This notebook accepts XML files up to 200 MB.")
    # Forbid DTD/entities, including UTF-16/32 encodings, before parsing any XML.
    if re.search(br"<!\s*(DOCTYPE|ENTITY)\b", data.replace(b"\x00", b""), re.I):
        raise FBMPError("DTD/entity declarations are not supported in FBMP result XML.")
    try:
        root = ET.fromstring(data)
    except ET.ParseError as exc:
        raise FBMPError(f"The file is not valid XML: {exc}") from None
    for e in root.iter():
        e.tag = e.tag.rsplit("}", 1)[-1]  # tolerate an XML namespace
    if root.tag != "FB-MULTIPIER_MODEL_DATA":
        raise FBMPError(f"Expected FB-MULTIPIER_MODEL_DATA; found {root.tag!r}.")

    source = str(filename).replace("\\", "/").split("/")[-1]
    raw, native, cases, warnings = [], [], [], []
    missing = Counter()
    case_lookup = {}
    for sub in root.findall("./MODEL_INFO/SUBSTRUCTURE"):
        sid = sub.get("number", "")
        for case in sub.findall("./LOAD_CASE_RESULTS/LOAD_CASE"):
            if not sid or not case.get("number"):
                raise FBMPError("Result substructure or load case has no number.")
            base = {"Source": source, "Substructure": sid, "Load_case": case.get("number", ""),
                    "Combination": case.get("combination", ""), "Limit_state": case.get("limitstate", "")}
            lookup_key = (sid, base["Load_case"], base["Combination"])
            previous = case_lookup.get(lookup_key)
            if previous is not None and previous != base["Limit_state"]:
                raise FBMPError(f"Conflicting limit states for case {lookup_key}.")
            case_lookup[lookup_key] = base["Limit_state"]
            steps = case.findall("./TIME_STEP")
            if not steps:
                steps = [case]
            for step in steps:
                ctx = {**base, "Time_step": step.get("number", "") if step is not case else ""}
                before = len(raw)
                for block in step:
                    if block.tag not in ("PILE_INTERNAL_FORCES", "STRUCTURE_INTERNAL_FORCES"):
                        if block.tag.endswith("INTERNAL_FORCES"):
                            warnings.append(f"Unsupported force block {block.tag} in substructure {sid}, case {base['Load_case']}.")
                        continue

                    def walk(node, groups):
                        if node.tag == "ELEMENT":
                            eid = node.get("elem_number", "")
                            local = node.get("number", "")
                            if not (eid or local):
                                raise FBMPError("An internal-force element has no identifier.")
                            family = groups[0][0] if groups else block.tag.replace("_INTERNAL_FORCES", "")
                            member = "/".join(f"{tag} {ident}" for tag, ident in groups if ident) or "all"
                            group_path = "/".join(f"{tag}[{ident}]" if ident else tag for tag, ident in groups)
                            ends_found = []
                            for end in node:
                                match = re.search(r"(?:^|_)ELEMENT_([IJ])_END$", end.tag)
                                if not match:
                                    continue
                                end_name = match.group(1)
                                ends_found.append(end_name)
                                row = {**ctx, "Member_type": family, "Member": member, "Group_path": group_path,
                                       "Element": eid or local, "Local_element": local, "End": end_name,
                                       "Node": end.get("node_" + end_name.lower(), ""), "Row_id": len(raw) + 1}
                                if not row["Node"]:
                                    raise FBMPError(f"Element {eid or local} end {end_name} has no node identifier.")
                                for comp, tag in FORCES.items():
                                    matches = end.findall(tag)
                                    if len(matches) > 1:
                                        raise FBMPError(f"Duplicate {tag} at element {eid or local}, end {end_name}.")
                                    val = matches[0] if matches else None
                                    row[comp], row[comp + "_unit"] = None, ""
                                    if val is None or not (val.text or "").strip():
                                        missing[(family, comp)] += 1
                                    else:
                                        row[comp] = _number(val.text, f"case {ctx['Load_case']}, element {eid or local}, end {end_name}, {tag}")
                                        row[comp + "_unit"] = val.get("units", "").strip() or "UNSPECIFIED"
                                if any(row[c] is not None for c in COMPONENTS):
                                    raw.append(row)
                                else:
                                    warnings.append(f"No requested forces at element {eid or local} end {end_name} in case {ctx['Load_case']}.")
                            if sorted(ends_found) != ["I", "J"]:
                                warnings.append(f"Element {eid or local}, case {ctx['Load_case']}: exported ends are {ends_found}; expected I and J.")
                            return
                        for child in node:
                            walk(child, groups + ([(child.tag, _identity(child))] if child.tag != "ELEMENT" else []))
                    walk(block, [])
                cases.append({**ctx, "End_force_rows": len(raw) - before})

    for sub in root.findall("./MODEL_INFO/SUBSTRUCTURE"):
        sid = sub.get("number", "")
        for summary in sub.findall("./OUTPUT_SUMMARY"):
            for block in summary:
                # Soil reactions, displacements, plate stresses, and capacities are not member forces.
                if block.tag != "PILE_MAX" and not (block.tag.startswith("STRUCTURE_") and block.tag.endswith("_MAX")):
                    continue
                family = "PILE" if block.tag == "PILE_MAX" else block.tag[len("STRUCTURE_"):-len("_MAX")]
                for item in block.findall("./MAX_ITEM"):
                    label = " ".join(item.get("item", "").lower().split())
                    parts = label.split(" ", 1)
                    if len(parts) != 2 or parts[0] not in ("min", "max") or parts[1] not in ITEM_MAP:
                        continue
                    value = item.find("ITEM_VALUE")
                    if value is None:
                        raise FBMPError(f"No ITEM_VALUE for {block.tag}: {label}.")
                    lc, combo = _text(item, "LOAD_CASE"), _text(item, "COMBINATION")
                    native.append({"Source": source, "Substructure": sid, "Member_type": family,
                                   "Summary_block": block.tag, "Component": ITEM_MAP[parts[1]], "Extreme": parts[0],
                                   "Value": _number(value.text, f"{block.tag}/{label}"),
                                   "Unit": value.get("units", "").strip() or "UNSPECIFIED", "Load_case": lc,
                                   "Combination": combo, "Limit_state": case_lookup.get((sid, lc, combo), ""),
                                   "Pile": _text(item, "PILE"), "Element": _text(item, "ELEMENT"), "Node": _text(item, "NODE"),
                                   "XML_item": item.get("item", "")})
    if not raw and not native:
        raise FBMPError("No member-force results or moment/shear/torsion OUTPUT_SUMMARY found. Export an analyzed FBMP model with XML result printing enabled.")
    seen = set()
    for row in raw:
        key = tuple(row[k] for k in ["Substructure", "Member_type", "Group_path", "Element", "End", "Node", "Load_case", "Combination", "Time_step"])
        if key in seen:
            raise FBMPError(f"Duplicate element-end result identity: {key}. Results were not silently combined.")
        seen.add(key)
    for (family, comp), count in missing.items():
        warnings.append(f"{family}: {count} exported end records lack {comp}. Missing values remain blank and are excluded from that component's envelope.")
    for comp in FORCES:
        units = sorted({row[comp + "_unit"] for row in raw if row[comp] is not None})
        if len(units) > 1 or "UNSPECIFIED" in units:
            warnings.append(f"{comp} unit labels: {units}. Results are separated by unit; no conversion is performed.")
    if not raw:
        warnings.append("Summary-only XML: detailed element locations, filters, concurrent forces, and plots are unavailable.")
    if not native:
        warnings.append("No supported native OUTPUT_SUMMARY envelopes were found. Only original XML end-force envelopes are available.")
    metadata = {"tool_version": VERSION, "source": source, "sha256": hashlib.sha256(data).hexdigest(),
                "fbmp_version": _text(root, "PROJECT_INFO/VERSION_NUMBER"),
                "project": _text(root, "PROJECT_INFO/PROJECT_NAME"), "unit_system": _text(root, "CONTROL_INFO/UNITS"),
                "analysis": _text(root, "CONTROL_INFO/ANALYSIS"),
                "extracted_utc": datetime.now(timezone.utc).isoformat(), "raw_end_rows": len(raw),
                "native_summary_items": len(native)}
    return {"raw": raw, "native": native, "cases": cases, "warnings": list(dict.fromkeys(warnings)), "metadata": metadata}


def native_envelopes(result):
    groups = defaultdict(dict)
    for row in result["native"]:
        key = tuple(row[k] for k in ("Source", "Substructure", "Member_type", "Summary_block", "Component", "Unit"))
        if row["Extreme"] in groups[key]:
            raise FBMPError(f"Duplicate native summary extreme: {key}, {row['Extreme']}.")
        groups[key][row["Extreme"]] = row
    out = []
    for key, pair in groups.items():
        row = dict(zip(("Source", "Substructure", "Member_type", "Summary_block", "Component", "Unit"), key))
        for which in ("min", "max"):
            original = pair.get(which, {})
            row[which.capitalize()] = original.get("Value")
            for field in ("Load_case", "Combination", "Limit_state", "Pile", "Element", "Node"):
                row[f"{which}_{field}"] = original.get(field, "")
        if len(pair) == 2:
            if row["Min"] > row["Max"]:
                raise FBMPError(f"Native summary minimum exceeds maximum: {key}.")
            best = max(pair.values(), key=lambda r: abs(r["Value"]))
            row["Abs_max"] = abs(best["Value"])
            row["Abs_governing_signed"] = best["Value"]
            row["Abs_governors"] = "; ".join(
                f"{r['Extreme']}: LC {r['Load_case']}, combination {r['Combination']}" for r in pair.values()
                if abs(r["Value"]) == row["Abs_max"])
            row["Status"] = "Complete min/max pair"
        else:
            row.update(Abs_max=None, Abs_governing_signed=None, Abs_governors="", Status="Incomplete min/max pair")
        out.append(row)
    return out


def end_envelopes(rows, scope="end", by_limit_state=False):
    """Envelope components independently, retaining a real concurrent governing record."""
    if scope not in ("end", "member"):
        raise ValueError("scope must be end or member")
    fields = ["Source", "Substructure", "Member_type", "Group_path", "Member"]
    if scope == "end":
        fields += ["Element", "Local_element", "End", "Node"]
    if by_limit_state:
        fields += ["Limit_state"]
    groups = defaultdict(list)
    for row in rows:
        for comp in COMPONENTS:
            if row[comp] is not None:
                groups[tuple(row[k] for k in fields) + (comp, row[comp + "_unit"])].append(row)
    result = []
    for key, candidates in groups.items():
        comp, unit = key[-2:]
        values = [r[comp] for r in candidates]
        for extreme, target in (("min", min(values)), ("max", max(values)), ("abs", max(map(abs, values)))):
            tied = [r for r in candidates if (abs(r[comp]) if extreme == "abs" else r[comp]) == target]
            governor = tied[0]  # deterministic: first XML record; every tie is linked below
            out = dict(zip(fields, key[:-2]))
            out.update(Component=comp, Unit=unit, Extreme=extreme, Value=target,
                       Governing_signed=governor[comp], Tie_count=len(tied),
                       Governing_row_ids=", ".join(str(r["Row_id"]) for r in tied),
                       Available_records=len(candidates))
            for name in ("Row_id", "Load_case", "Combination", "Limit_state", "Time_step", "Element", "Local_element", "End", "Node"):
                out["Governing_" + name] = governor[name]
            for component in FORCES:
                out["Concurrent_" + component] = governor[component]
                out["Concurrent_" + component + "_unit"] = governor[component + "_unit"]
            result.append(out)
    return result


def limit_state_family(name):
    """Classify exported labels without combining distinct Service or Strength states."""
    label = str(name or "").strip().upper()
    if label.startswith("STRENGTH"):
        return "Strength"
    if label.startswith("SERVICE"):
        return "Service"
    return "Other / unspecified"


def limit_state_table(result):
    """One readable min/max/absolute row per member, exact limit state, component and unit."""
    pairs = defaultdict(dict)
    keys = ("Source", "Substructure", "Member_type", "Group_path", "Member", "Limit_state", "Component", "Unit")
    for record in end_envelopes(result["raw"], scope="member", by_limit_state=True):
        pairs[tuple(record[k] for k in keys)][record["Extreme"]] = record
    output = []
    for key, pair in pairs.items():
        row = dict(zip(keys, key))
        row["Limit_state_family"] = limit_state_family(row["Limit_state"])
        row["Basis"] = "Original XML element-local I/J end forces"
        for extreme, prefix in (("min", "Min"), ("max", "Max"), ("abs", "Abs_max")):
            record = pair[extreme]
            row[prefix] = record["Value"]
            row[prefix + "_signed"] = record["Governing_signed"]
            for field in ("Load_case", "Combination", "Time_step", "Element", "End", "Node", "Row_id"):
                row[prefix + "_" + field] = record["Governing_" + field]
            row[prefix + "_tie_count"] = record["Tie_count"]
            row[prefix + "_row_ids"] = record["Governing_row_ids"]
        row["Available_records"] = pair["min"]["Available_records"]
        output.append(row)
    return sorted(output, key=lambda r: (_nat(r["Substructure"]), r["Member_type"] != "PIER_CAP",
                  r["Member_type"], _nat(r["Group_path"]), _nat(r["Limit_state"]),
                  list(COMPONENTS).index(r["Component"]), r["Unit"]))


def comparison(result):
    """Compare magnitudes only; a difference is not corrected or silently suppressed."""
    output = []
    for row in native_envelopes(result):
        comp = row["Component"]
        subset = [r for r in result["raw"] if r["Substructure"] == row["Substructure"]
                  and r["Member_type"] == row["Member_type"] and r[comp] is not None
                  and r[comp + "_unit"] == row["Unit"]]
        raw_abs = max((abs(r[comp]) for r in subset), default=None)
        reported = row["Abs_max"]
        status = "Not comparable"
        if reported is not None and raw_abs is not None:
            status = "Magnitude matches" if math.isclose(reported, raw_abs, rel_tol=1e-10, abs_tol=1e-10) else "DIFFERENT MAGNITUDE"
        output.append({"Substructure": row["Substructure"], "Member_type": row["Member_type"], "Component": comp,
                       "Unit": row["Unit"], "Reported_abs_max": reported, "All_exported_ends_abs_max": raw_abs, "Comparison": status})
    return output


def coverage(result):
    buckets = defaultdict(list)
    for r in result["raw"]:
        key = tuple(r[k] for k in ("Substructure", "Member_type", "Member", "Group_path", "Load_case", "Combination", "Limit_state", "Time_step"))
        buckets[key].append(r)
    out = []
    for key, rows in buckets.items():
        row = dict(zip(("Substructure", "Member_type", "Member", "Group_path", "Load_case", "Combination", "Limit_state", "Time_step"), key))
        row["End_records"] = len(rows)
        for comp in COMPONENTS:
            row["Available_" + comp] = sum(r[comp] is not None for r in rows)
        out.append(row)
    return out


def diagnostics(result):
    messages = list(result["warnings"])
    for r in result["cases"]:
        if r["End_force_rows"] == 0:
            messages.append(f"Substructure {r['Substructure']}, case {r['Load_case']}, time step {r['Time_step']}: no detailed force rows exported.")
    inventories = defaultdict(dict)
    for r in result["raw"]:
        owner = (r["Substructure"], r["Member_type"], r["Group_path"])
        case = (r["Load_case"], r["Combination"], r["Time_step"])
        inventories[owner].setdefault(case, set()).update((r["Element"], r["End"], r["Node"], comp) for comp in COMPONENTS if r[comp] is not None)
    for owner, cases in inventories.items():
        expected_cases = {(r["Load_case"], r["Combination"], r["Time_step"]) for r in result["cases"] if r["Substructure"] == owner[0]}
        if len({frozenset(v) for v in cases.values()}) > 1 or set(cases) != expected_cases:
            messages.append(f"Unequal force coverage across cases/time steps for {owner}. Envelopes include available records only; check coverage.csv.")
    for row in native_envelopes(result):
        if row["Status"].startswith("Incomplete"):
            messages.append(f"Incomplete native min/max pair: {row['Member_type']} {row['Component']}; absolute maximum is unavailable.")
    differences = [r for r in comparison(result) if r["Comparison"] == "DIFFERENT MAGNITUDE"]
    if differences:
        messages.append("Reported envelopes and all exported element ends differ in magnitude for: " +
                        ", ".join(f"substructure {r['Substructure']} {r['Member_type']} {r['Component']}" for r in differences) +
                        ". This may involve a different reported member scope or an export inconsistency. The tool preserves both and does not infer a correction.")
    return list(dict.fromkeys(messages))


def csv_bytes(rows):
    if not rows:
        return b"\xef\xbb\xbfStatus\r\nNo records available\r\n"
    stream = io.StringIO(newline="")
    columns = list(dict.fromkeys(k for row in rows for k in row))
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    # Protect spreadsheet viewers from interpreting XML labels as formulas; numeric forces stay numeric.
    for row in rows:
        writer.writerow({k: ("'" + v if isinstance(v, str) and v.lstrip().startswith(("=", "+", "-", "@")) else v) for k, v in row.items()})
    return stream.getvalue().encode("utf-8-sig")


def export_tables(result):
    table = limit_state_table(result)
    return {"limit_state_envelope_table": table,
            "strength_envelopes": [r for r in table if r["Limit_state_family"] == "Strength"],
            "service_envelopes": [r for r in table if r["Limit_state_family"] == "Service"],
            "reported_fbmp_envelopes": native_envelopes(result), "reported_summary_source": result["native"],
            "element_end_envelopes": end_envelopes(result["raw"]),
            "member_end_force_envelopes": end_envelopes(result["raw"], scope="member"),
            "member_end_force_by_limit_state": end_envelopes(result["raw"], scope="member", by_limit_state=True),
            "raw_element_end_forces": result["raw"], "load_cases": result["cases"],
            "coverage": coverage(result), "reported_vs_exported_magnitudes": comparison(result)}


def package_results(result, selected_rows=None, selection=None):
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as z:
        for name, rows in export_tables(result).items():
            z.writestr(name + ".csv", csv_bytes(rows))
        if selected_rows is not None:
            z.writestr("selected_element_end_envelopes.csv", csv_bytes(end_envelopes(selected_rows)))
            z.writestr("selected_raw_forces.csv", csv_bytes(selected_rows))
            z.writestr("selection.json", json.dumps(selection or {}, indent=2))
        z.writestr("metadata.json", json.dumps(result["metadata"], indent=2))
        z.writestr("checks.txt", "\n".join(diagnostics(result)) or "No parsing or coverage issues detected.")
        z.writestr("README.txt", f"FBMP envelope extractor {VERSION}\nSource: {result['metadata']['source']}\n\n{SIGN_NOTE}\n\n{METHOD_NOTE}\n\n"
                   "Start with limit_state_envelope_table.csv for the same Strength/Service tables displayed in the notebook. These are computed from original XML element-end forces, separately for each exact limit state and member.\n"
                   "strength_envelopes.csv and service_envelopes.csv contain the corresponding table rows. Service I, II, III, etc. remain separate when present. An absent state is not fabricated or scaled from Strength.\n"
                   "Start with reported_fbmp_envelopes.csv for the envelopes stored by FBMP. The reported summary spans the original export and cannot be re-filtered by its governing load cases.\n"
                   "element_end_envelopes.csv groups by substructure, member, element, I/J end, node, component, and unit. Cases/combinations/time steps are enveloped within each group.\n"
                   "member_end_force_envelopes.csv also envelopes over elements/ends within the XML member group. The by_limit_state version keeps limit states separate.\n"
                   "Raw row IDs link every detailed governor and tie to raw_element_end_forces.csv. Concurrent values correspond only to the first tied record. Ties use equality at the XML's exported precision.\n"
                   "selected_* files, when included, reflect the explorer's member/limit-state/case selection and include all five components. Other files contain the full export.\n"
                   "Empty values are unavailable, not zero. P is axial force included only as concurrent context. No SI/English conversion is applied.\n"
                   "See reported_vs_exported_magnitudes.csv and checks.txt for differences in the source.\n"
                   "BSI sign-convention background: https://bsi.ce.ufl.edu/newsletter/BSI-Newsletter-Fall-2017.pdf (page 10).\n")
    return buffer.getvalue()


"""Colab notebook interface. The core extractor above is embedded, not imported."""
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, display, clear_output

STYLE = """<style>
.fbmp-box{font-family:system-ui,sans-serif;color:#183348;max-width:1180px;padding:18px 22px;border:1px solid #dce5ea;border-radius:12px;background:#fff;margin:10px 0}
.fbmp-box h2{margin:0 0 8px;font-size:23px}.fbmp-box h3{margin:12px 0 6px;font-size:17px}
.fbmp-eyebrow{color:#537185;font-weight:700;letter-spacing:1.5px;font-size:11px;text-transform:uppercase}
.fbmp-muted{color:#526777;font-size:13px;line-height:1.5}.fbmp-note{border-left:4px solid #d79934;background:#fff8eb;padding:12px 15px;margin:12px 0;line-height:1.5}
.fbmp-table{border-collapse:collapse;width:100%;font-size:12px}.fbmp-table th{background:#173a51;color:white;text-align:left;padding:10px;white-space:nowrap}.fbmp-table td{padding:9px 10px;border-bottom:1px solid #e4ebf0;white-space:nowrap}.fbmp-table tbody tr:nth-child(even){background:#f4f8fa}
.fbmp-scroll{overflow-x:auto;max-height:520px;overflow-y:auto}.fbmp-scroll thead{position:sticky;top:0}.fbmp-metrics{display:flex;gap:12px;margin:14px 0;flex-wrap:wrap}.fbmp-metric{padding:12px 18px;background:#eef5f8;border-radius:8px;min-width:130px}.fbmp-metric b{font-size:24px;display:block}
</style>"""


def _format(value):
    if value is None or pd.isna(value):
        return "—"
    if isinstance(value, (float, int)):
        return f"{value:.6g}"
    return str(value)


def show_table(rows, columns=None, max_rows=250):
    if not rows:
        display(HTML('<p class="fbmp-muted">No records available for this view.</p>'))
        return
    frame = pd.DataFrame(rows)
    if columns:
        frame = frame.reindex(columns=columns)
    frame = frame.head(max_rows)
    table = frame.to_html(index=False, escape=True, na_rep="—", classes="fbmp-table", border=0,
                          float_format=lambda x: f"{x:.6g}")
    display(HTML('<div class="fbmp-scroll">' + table + '</div>'))
    if len(rows) > max_rows:
        display(HTML(f'<p class="fbmp-muted">Showing {max_rows:,} of {len(rows):,} rows. Downloads contain every row.</p>'))


def compact_native(rows):
    def gov(r, prefix):
        txt = f"LC {r[prefix + '_Load_case']} / C{r[prefix + '_Combination']}"
        if r[prefix + "_Pile"]:
            txt += " / pile " + r[prefix + "_Pile"]
        return txt
    return [{"Component": r["Component"], "Units": r["Unit"], "Minimum": r["Min"], "Min case / combo": gov(r, "min"),
             "Min limit state": r["min_Limit_state"], "Maximum": r["Max"], "Max case / combo": gov(r, "max"),
             "Max limit state": r["max_Limit_state"], "Max |value|": r["Abs_max"]} for r in rows]


def display_limit_state_tables(result, table):
    """Display ordinary notebook HTML tables before the interactive explorer."""
    display(HTML('<div class="fbmp-box"><h3>Strength &amp; Service envelope tables</h3>'
                 '<p class="fbmp-muted">All five components are shown below, separately for each exported limit state and member. '
                 'These tables use the original XML I/J end-force signs and the full file; explorer filters apply only to the explorer. '
                 'LC = load case; C = combination. Governing locations and tied records are included in the matching CSVs.</p>'
                 '<p class="fbmp-muted"><b>Force basis:</b> calculated from detailed element-end records. These values can differ '
                 'from FBMP\'s all-case summary shown in the Original FBMP summary tab.</p></div>'))
    if not result["raw"]:
        display(HTML('<div class="fbmp-note">This XML contains no detailed element-end forces. Separate Strength/Service '
                     'envelopes cannot be reconstructed from the overall summary. Export the analyzed cases with detailed forces.</div>'))
        return
    for category in ("Strength", "Service", "Other / unspecified"):
        rows = [r for r in table if r["Limit_state_family"] == category]
        if category == "Other / unspecified" and not rows:
            continue
        display(HTML(f'<h2>{html.escape(category)} envelopes</h2>'))
        exported_cases = [r for r in result["cases"] if limit_state_family(r["Limit_state"]) == category]
        if not rows:
            message = (f"{category} load cases are listed in the XML, but no detailed member-force records are available."
                       if exported_cases else f"No {category.lower()} load cases were exported in this XML.")
            display(HTML('<div class="fbmp-note">' + html.escape(message) + ' No values are generated from other limit states.</div>'))
            continue
        case_labels = sorted({(r["Limit_state"], r["Load_case"], r["Combination"]) for r in exported_cases}, key=lambda x: (_nat(x[0]), _nat(x[1])))
        case_text = "; ".join(f"{ls or 'Unspecified'}: LC {lc}, C{combo or '—'}" for ls, lc, combo in case_labels)
        display(HTML('<p class="fbmp-muted">Included cases: ' + html.escape(case_text) + '</p>'))
        owners = list(dict.fromkeys((r["Substructure"], r["Member_type"], r["Group_path"], r["Member"]) for r in rows))
        for sid, family, path, member_name in owners:
            selected = [r for r in rows if (r["Substructure"], r["Member_type"], r["Group_path"], r["Member"]) == (sid, family, path, member_name)]
            name = member_name if member_name != "all" else path.replace("_", " ").title()
            display(HTML(f'<h3>Substructure {html.escape(sid)} · {html.escape(name)}</h3>'))
            compact = [{"Limit state": r["Limit_state"] or "Unspecified", "Component": r["Component"], "Unit": r["Unit"],
                        "Minimum": r["Min"], "Min LC / C": f"{r['Min_Load_case']} / {r['Min_Combination']}",
                        "Maximum": r["Max"], "Max LC / C": f"{r['Max_Load_case']} / {r['Max_Combination']}",
                        "Max |value|": r["Abs_max"], "Abs LC / C": f"{r['Abs_max_Load_case']} / {r['Abs_max_Combination']}"}
                       for r in selected]
            show_table(compact, max_rows=len(compact))


def overview_html(result):
    meta = result["metadata"]
    cases = {(r["Substructure"], r["Load_case"], r["Combination"]) for r in result["cases"]}
    elements = {(r["Substructure"], r["Group_path"], r["Element"]) for r in result["raw"]}
    return STYLE + f'''<div class="fbmp-box"><div class="fbmp-eyebrow">FB-MultiPier · XML results</div>
    <h2>Moment, shear &amp; torsion envelopes</h2>
    <div class="fbmp-muted">{html.escape(meta['source'])} &nbsp; · &nbsp; {html.escape(meta['project'])} &nbsp; · &nbsp; FBMP {html.escape(meta['fbmp_version'])}</div>
    <div class="fbmp-metrics"><div class="fbmp-metric"><b>{len(cases)}</b>analyzed case / combinations</div>
    <div class="fbmp-metric"><b>{len(elements)}</b>exported elements</div>
    <div class="fbmp-metric"><b>{len(result['raw']):,}</b>element-end records</div></div>
    <p class="fbmp-muted">M2 / M3: moments about local axes 2 / 3 · V2 / V3: shears in local directions 2 / 3 · T: torsion.<br>
    Units and force signs are retained from the XML. Each file is processed independently.</p></div>'''


def make_app(result):
    state = {"selected": list(result["raw"]), "selection": {}, "busy": False, "error": None}
    display(HTML(overview_html(result)))
    native_rows = native_envelopes(result)
    messages = diagnostics(result)
    table = limit_state_table(result)
    state["limit_state_table"] = table
    display_limit_state_tables(result, table)
    if any(r["Comparison"] == "DIFFERENT MAGNITUDE" for r in comparison(result)):
        display(HTML('<div class="fbmp-note"><b>Source discrepancy:</b> the detailed force magnitudes and original '
                     'FBMP summary differ for some components. The tables above preserve the detailed records. '
                     'See Data checks for the comparison with the original summary.</div>'))
    tabs_out = [widgets.Output() for _ in range(3)]
    with tabs_out[0]:
        display(HTML('<div class="fbmp-box"><h3>Original FBMP summary</h3><p class="fbmp-muted">'
                     'Copied from OUTPUT_SUMMARY for the full export. The explorer filters do not change these reported envelopes. '
                     'Each component can have a different governing case. Service cases may not govern this overall summary; '
                     'their separate envelopes are in the Service tables above.</p></div>'))
        if not native_rows:
            display(HTML('<div class="fbmp-note">No reported envelope summary is present. Use the detailed explorer for original XML end-force envelopes.</div>'))
        owners = sorted({(r["Substructure"], r["Member_type"]) for r in native_rows}, key=lambda x: (x[1] != "PIER_CAP", _nat(x[0]), x[1]))
        for sid, family in owners:
            display(HTML(f'<h3>Substructure {html.escape(sid)} · {html.escape(family.replace("_", " ").title())}</h3>'))
            show_table(compact_native([r for r in native_rows if r["Substructure"] == sid and r["Member_type"] == family]))
        if any(r["Comparison"] == "DIFFERENT MAGNITUDE" for r in comparison(result)):
            display(HTML('<div class="fbmp-note"><b>Source discrepancy detected.</b> Some reported envelope magnitudes differ from the detailed XML block. '
                         'Both are preserved. Open <b>Data checks</b> for the comparison.</div>'))
        display(HTML('<p class="fbmp-muted">The summary does not provide element/node locations for every result. Missing locations are left blank in the downloads.</p>'))

    style = {"description_width": "100px"}
    layout = widgets.Layout(width="360px")
    sub = widgets.Dropdown(description="Substructure", style=style, layout=layout)
    family = widgets.Dropdown(description="Member type", style=style, layout=layout)
    member = widgets.Dropdown(description="Member", style=style, layout=layout)
    limit = widgets.Dropdown(description="Limit state", style=style, layout=layout)
    cases = widgets.SelectMultiple(description="Load cases", style=style, layout=widgets.Layout(width="740px", height="115px"))
    component = widgets.Dropdown(options=[(f"{c} · {LABELS[c]}", c) for c in COMPONENTS], value="M3", description="Component", style=style, layout=layout)
    detail_out = widgets.Output()
    selected_button = widgets.Button(description="Download selected CSV", icon="download", button_style="info", layout=widgets.Layout(width="220px"))
    file_out = widgets.Output()

    def base_rows():
        return [r for r in result["raw"] if r["Substructure"] == sub.value and r["Member_type"] == family.value
                and r["Group_path"] == member.value]

    def limit_rows():
        rows = base_rows()
        if limit.value == "@ALL":
            return rows
        if limit.value == "@STRENGTH":
            return [r for r in rows if r["Limit_state"].upper().startswith("STRENGTH")]
        if limit.value == "@SERVICE":
            return [r for r in rows if r["Limit_state"].upper().startswith("SERVICE")]
        return [r for r in rows if r["Limit_state"] == limit.value]

    def draw():
        if state["busy"]:
            return
        state["error"] = None
        rows = [r for r in limit_rows() if (r["Load_case"], r["Combination"], r["Limit_state"]) in cases.value]
        state["selected"] = rows
        state["selection"] = {"substructure": sub.value, "member_type": family.value, "member_group": member.value,
                              "limit_state_filter": limit.value, "cases": list(cases.value), "plotted_component": component.value,
                              "basis": "Original XML element-local I/J end forces"}
        selected_button.disabled = not bool(rows)
        with detail_out:
            clear_output(wait=True)
            if not rows:
                print("Select at least one load case with exported force results.")
                return
            envelope = end_envelopes(rows)
            selected = [r for r in envelope if r["Component"] == component.value]
            state["envelope"] = envelope
            display(HTML(f'<p class="fbmp-muted">{len(rows):,} end records selected. Original XML end-force signs; all selected time steps included. '
                         'At shared nodes, separate I/J results remain separate.</p>'))
            if not selected:
                print("The selected component is unavailable in these records.")
                return
            stats = end_envelopes(rows, scope="member")
            show_table([{"Component": r["Component"], "Units": r["Unit"], "Extreme": r["Extreme"], "Value": r["Value"],
                         "Signed governor": r["Governing_signed"], "Load case": r["Governing_Load_case"], "Combination": r["Governing_Combination"],
                         "Element": r["Governing_Element"], "End": r["Governing_End"], "Node": r["Governing_Node"], "Ties": r["Tie_count"]}
                        for r in stats if r["Component"] == component.value])
            for unit in sorted({r["Unit"] for r in selected}):
                points = [r for r in selected if r["Unit"] == unit]
                elements = sorted({r["Element"] for r in points}, key=_nat)
                positions = {e: i for i, e in enumerate(elements)}
                fig, axes = plt.subplots(1, 2, figsize=(13, 3.9), sharey=True, constrained_layout=True)
                for ax, end_name in zip(axes, ("I", "J")):
                    for extreme, color in (("min", "#247b98"), ("max", "#c47137")):
                        pts = [r for r in points if r["End"] == end_name and r["Extreme"] == extreme]
                        ax.scatter([positions[r["Element"]] for r in pts], [r["Value"] for r in pts], s=22, color=color, label=extreme.capitalize(), alpha=.85)
                    ax.axhline(0, color="#b8c8d1", linewidth=.8)
                    ax.set_title(f"{component.value} · {end_name} end", loc="left", fontsize=12)
                    step = max(1, math.ceil(len(elements) / 12))
                    ticks = list(range(0, len(elements), step))
                    ax.set_xticks(ticks, [elements[i] for i in ticks], rotation=45)
                    ax.set_xlabel("Element ID (ordered identifiers; not physical distance)", fontsize=9)
                    ax.grid(axis="y", alpha=.15)
                    ax.spines[["top", "right"]].set_visible(False)
                    ax.legend(frameon=False, fontsize=9)
                axes[0].set_ylabel(f"Original XML end force [{unit}]")
                plt.show()
                plt.close(fig)
            compact = []
            pair_map = defaultdict(dict)
            for r in selected:
                pair_map[(r["Element"], r["End"], r["Node"], r["Unit"])][r["Extreme"]] = r
            for key in sorted(pair_map, key=lambda k: (_nat(k[0]), k[1], k[3])):
                p = pair_map[key]
                compact.append({"Element": key[0], "End": key[1], "Node": key[2], "Unit": key[3],
                                "Min": p["min"]["Value"], "Min LC": p["min"]["Governing_Load_case"], "Min combo": p["min"]["Governing_Combination"],
                                "Max": p["max"]["Value"], "Max LC": p["max"]["Governing_Load_case"], "Max combo": p["max"]["Governing_Combination"],
                                "Max |value|": p["abs"]["Value"]})
            show_table(compact)
            display(HTML('<p class="fbmp-muted">Downloads include all five components, concurrent forces, every tied governing row ID, and time-step IDs.</p>'))

    def reset_choices(level):
        state["busy"] = True
        try:
            if level <= 0:
                sub.options = sorted({r["Substructure"] for r in result["raw"]}, key=_nat)
                sub.value = sub.options[0]
            if level <= 1:
                family.options = sorted({r["Member_type"] for r in result["raw"] if r["Substructure"] == sub.value}, key=lambda x: (x != "PIER_CAP", x))
                family.value = family.options[0]
            if level <= 2:
                opts = {(r["Group_path"], r["Member"]) for r in result["raw"] if r["Substructure"] == sub.value and r["Member_type"] == family.value}
                member.options = [(name if name != "all" else path.replace("_", " "), path) for path, name in sorted(opts, key=lambda p: _nat(p[0]))]
                member.value = member.options[0][1]
            if level <= 3:
                limits = sorted({r["Limit_state"] for r in base_rows()})
                opts = [("All exported cases", "@ALL")]
                if any(v.upper().startswith("STRENGTH") for v in limits):
                    opts.append(("Strength cases (all)", "@STRENGTH"))
                if any(v.upper().startswith("SERVICE") for v in limits):
                    opts.append(("Service cases (all)", "@SERVICE"))
                limit.options = opts + [(v or "Unspecified", v) for v in limits]
                limit.value = "@ALL"
            if level <= 4:
                choices = sorted({(r["Load_case"], r["Combination"], r["Limit_state"]) for r in limit_rows()}, key=lambda k: (_nat(k[0]), _nat(k[1])))
                cases.options = [(f"LC {lc} · combination {combo or '—'} · {ls or 'unspecified'}", (lc, combo, ls)) for lc, combo, ls in choices]
                cases.value = tuple(choices)
        finally:
            state["busy"] = False
        draw()

    def on_level(level):
        def change(event):
            if not state["busy"]:
                reset_choices(level)
        return change

    def download(payload, suffix):
        with file_out:
            clear_output(wait=True)
            stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(result["metadata"]["source"]).stem)[:90] or "fbmp"
            folder = Path("/content/fbmp_envelopes") if Path("/content").exists() else Path.cwd() / "fbmp_envelopes"
            folder.mkdir(parents=True, exist_ok=True)
            target = folder / (stem + suffix)
            target.write_bytes(payload)
            try:
                from google.colab import files
                files.download(str(target))
            except ImportError:
                from IPython.display import FileLink
                display(FileLink(str(target)))
            print(f"Saved {target.name}")
            return target

    all_button = widgets.Button(description="Download all results (ZIP)", icon="download", button_style="success", layout=widgets.Layout(width="230px"))
    summary_button = widgets.Button(description="Download FBMP summary CSV", icon="download", layout=widgets.Layout(width="250px"), disabled=not bool(native_rows))
    strength_button = widgets.Button(description="Download Strength CSV", icon="download", layout=widgets.Layout(width="225px"),
                                     disabled=not any(r["Limit_state_family"] == "Strength" for r in table))
    service_button = widgets.Button(description="Download Service CSV", icon="download", button_style="info", layout=widgets.Layout(width="225px"),
                                    disabled=not any(r["Limit_state_family"] == "Service" for r in table))
    strength_button.on_click(lambda _: download(csv_bytes([r for r in table if r["Limit_state_family"] == "Strength"]), "_strength_envelopes.csv"))
    service_button.on_click(lambda _: download(csv_bytes([r for r in table if r["Limit_state_family"] == "Service"]), "_service_envelopes.csv"))
    all_button.on_click(lambda _: download(package_results(result, state["selected"] if result["raw"] else None, state["selection"]), "_envelopes.zip"))
    summary_button.on_click(lambda _: download(csv_bytes(native_rows), "_reported_envelopes.csv"))
    selected_button.on_click(lambda _: download(csv_bytes(end_envelopes(state["selected"])), "_selected_element_end_envelopes.csv"))
    with tabs_out[1]:
        display(HTML('<div class="fbmp-box"><h3>Element-end explorer</h3><p class="fbmp-muted">'
                     'Envelopes recalculated from the original XML I/J end forces for the selected cases. '
                     'These values can differ from the reported FBMP summary. Hold Ctrl / Cmd to select multiple cases.</p></div>'))
        if result["raw"]:
            display(widgets.VBox([widgets.HBox([sub, family]), widgets.HBox([member, limit]), cases, component, selected_button, detail_out]))
        else:
            print("Detailed element-end results are absent from this XML.")
    with tabs_out[2]:
        display(HTML('<h3>Source checks</h3>'))
        if messages:
            display(HTML('<div class="fbmp-note"><ul>' + "".join('<li>' + html.escape(m) + '</li>' for m in messages) + '</ul></div>'))
        else:
            print("No parsing or coverage issues detected.")
        show_table(comparison(result))
        display(HTML('<p class="fbmp-muted">A matching magnitude alone does not establish the same sign convention or member scope. '
                     'A different magnitude requires reviewing the FBMP export; this tool does not alter it.</p><h3>Case coverage</h3>'))
        show_table(result["cases"])
        display(HTML('<h3>Method</h3><p class="fbmp-muted">' + html.escape(SIGN_NOTE) + '</p><p class="fbmp-muted">' + html.escape(METHOD_NOTE) + '</p>'
                     '<p class="fbmp-muted">BSI documents sign adjustments in its design-oriented tables: '
                     '<a href="https://bsi.ce.ufl.edu/newsletter/BSI-Newsletter-Fall-2017.pdf" target="_blank">Fall 2017 newsletter, page 10</a>.</p>'))
    tabs = widgets.Tab(children=tabs_out)
    for i, title in enumerate(("Original FBMP summary", "Element-end explorer", "Data checks")):
        tabs.set_title(i, title)
    display(widgets.VBox([widgets.HBox([strength_button, service_button]), widgets.HBox([all_button, summary_button]), file_out, tabs]))
    if result["raw"]:
        sub.observe(on_level(1), names="value")
        family.observe(on_level(2), names="value")
        member.observe(on_level(3), names="value")
        limit.observe(on_level(4), names="value")
        cases.observe(lambda event: draw(), names="value")
        component.observe(lambda event: draw(), names="value")
        reset_choices(0)
    return {"state": state, "substructure": sub, "family": family, "member": member, "limit": limit,
            "cases": cases, "component": component, "tabs": tabs, "draw": draw, "download": download,
            "buttons": {"all": all_button, "summary": summary_button, "selected": selected_button,
                        "strength": strength_button, "service": service_button}}


def launch_fbmp(uploaded):
    parsed, errors = {}, []
    for name, payload in uploaded.items():
        try:
            result = read_fbmp(payload, name)
            native_envelopes(result)  # validate native pairs before opening any output
            parsed[name] = result
        except FBMPError as exc:
            errors.append(f"{name}: {exc}")
    if errors:
        display(HTML('<div class="fbmp-note">' + '<br>'.join(html.escape(e) for e in errors) + '</div>'))
    if not parsed:
        print("No usable XML files loaded. Run the upload cell again to choose an analyzed FBMP XML.")
        return {"results": parsed, "active": None, "errors": errors}
    outer = {"results": parsed, "active": None, "errors": errors}
    if len(parsed) == 1:
        # Keep the initial tables as ordinary cell output, outside widget Output containers.
        outer["active"] = make_app(next(iter(parsed.values())))
        return outer
    selector = widgets.Dropdown(options=list(parsed), description="XML file", layout=widgets.Layout(width="600px"))
    output = widgets.Output()
    def choose(event=None):
        with output:
            clear_output(wait=True)
            outer["active"] = make_app(parsed[selector.value])
    if len(parsed) > 1:
        display(selector)
    display(output)
    selector.observe(choose, names="value")
    choose()
    outer["file_selector"] = selector
    return outer


In [ ]:
#@title Upload XML and open the tool
from google.colab import files
from google.colab import output
output.enable_custom_widget_manager()
print("Choose an analyzed FB-MultiPier XML. Select more than one file to process them separately.")
FBMP_UPLOADS = files.upload()
FBMP_APP = launch_fbmp(FBMP_UPLOADS)


### Reading the downloads

- **limit_state_envelope_table.csv** — the same readable Strength/Service table rows shown in the notebook, with exact limit states kept separate and additional governing locations and tie IDs.
- **strength_envelopes.csv** and **service_envelopes.csv** — separate exports of those table rows; also available from their own download buttons.
- **reported_fbmp_envelopes.csv** — the original FBMP summary with min/max, governing cases, and maximum absolute magnitude. Start here for FBMP's reported envelopes.
- **element_end_envelopes.csv** — original XML end-force min/max/absolute envelopes across cases and time steps, at each element's I and J ends.
- **member_end_force_envelopes.csv** and **member_end_force_by_limit_state.csv** — detailed end-force extremes for each XML member group, with the latter separating the limit states.
- **raw_element_end_forces.csv** — the underlying records. `Row_id` links every detailed governing result and all its ties back to the source values.
- **selected_*.csv** — the current explorer selection, included in the ZIP when detailed records exist. The selection applies to all five components; the component dropdown controls the display only.
- **reported_vs_exported_magnitudes.csv**, **coverage.csv**, **checks.txt**, and **metadata.json** — source discrepancies, missing-data checks, record counts, and the source-file SHA-256.

Each component is enveloped independently. Separate maxima do not form a simultaneous load vector. `Concurrent_*` columns contain the forces at the first tied governing record, and all tied source row IDs are listed. Missing values stay blank. Original unit labels stay attached to the values; no unit conversion, sign transformation, load factoring, or new structural analysis is performed. No interior peaks between exported element ends are inferred. Plot positions are ordered element IDs, not physical distances. The source file has finite printed precision, which limits the precision of the extraction.

The parser supports the supplied FBMP XML layout, including repeated `MODEL_INFO` blocks, multiple substructures, cases, combinations and time steps. It recognizes pile and structure member-force blocks plus their native force summaries. Missing results are reported; soil reactions, displacements, capacities, and plate stresses are excluded.

For background on FBMP's design-table sign adjustments, see [BSI's Fall 2017 newsletter, page 10](https://bsi.ce.ufl.edu/newsletter/BSI-Newsletter-Fall-2017.pdf).

To load a different model, rerun **Upload XML and open the tool**. Downloads reflect the current file only.
